# Mutual Fund Analytics — Performance & Risk Analytics
## Capstone Project | Day 4 | Financial Metrics & Rankings

**Author:** Senior Financial Data Analyst
**Database:** `mutual_funds.db` (SQLite)
**Benchmarks:** Nifty 100 (`^CNX100`) & Nifty 50 (`^NSEI`)
**Metrics Calculated:** CAGR (1Y, 3Y, 5Y), Sharpe Ratio, Sortino Ratio, CAPM Beta, Annualized Alpha, Maximum Drawdown, Tracking Error, and Weighted Fund Scorecard.

---
### Analytics Workflow
1. **Setup & Imports**: Initialize environment and paths.
2. **Data Loading**: Read processed CSV files (`nav_history_clean.csv`, `scheme_performance_clean.csv`, `fund_master_clean.csv`, `nifty100.csv`, `nifty50.csv`).
3. **Daily Returns Calculation**: Compute daily return series for schemes and benchmark.
4. **CAGR Analysis**: Annualized compounding returns (1Y, 3Y, 5Y).
5. **Sharpe & Sortino Ratios**: Risk-adjusted returns relative to risk-free rate of 6.5%.
6. **Alpha & Beta (CAPM)**: Regress scheme excess returns against Nifty 100.
7. **Maximum Drawdown**: Compute historical peaks and maximum peak-to-trough losses.
8. **Tracking Error**: Standard deviation of excess active returns.
9. **Weighted Fund Scorecard**: Compute score using multi-factor ranks.
10. **Benchmark Comparison Visualizations**: Interactive and static plots.


## 1. Setup & Environment Configuration

In [1]:
# ============================================================
# SETUP — Imports, Directory Creation, Parameters
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path

import numpy  as np
import pandas as pd
import scipy.stats as stats

# Plotly & Seaborn
import plotly.express       as px
import plotly.graph_objects as go
import seaborn  as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- Paths ----
_cwd = Path().resolve()
if (_cwd / "mutual_funds.db").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "mutual_funds.db").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR   = PROJECT_ROOT / "reports"
CHARTS_DIR    = REPORTS_DIR / "charts"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Constants ----
RISK_FREE_RATE_ANN = 0.065   # 6.5% Annual Risk-Free Rate
TRADING_DAYS_YEAR  = 252     # Standard annualization factor
PLOTLY_TEMPLATE    = "plotly_dark"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Processed Data: {PROCESSED_DIR}")
print(f"Reports Path: {REPORTS_DIR}")
print("Setup complete.")


Project Root: C:\Users\dhile\OneDrive\Documents\bluestock_capstone _project
Processed Data: C:\Users\dhile\OneDrive\Documents\bluestock_capstone _project\data\processed
Reports Path: C:\Users\dhile\OneDrive\Documents\bluestock_capstone _project\reports
Setup complete.


## 2. Data Loading & Date Synchronization

In [2]:
# ============================================================
# DATA LOADING
# ============================================================
print("Loading datasets...")

nav_df = pd.read_csv(PROCESSED_DIR / "nav_history_clean.csv", parse_dates=["nav_date"])
perf_df = pd.read_csv(PROCESSED_DIR / "scheme_performance_clean.csv")
master_df = pd.read_csv(PROCESSED_DIR / "fund_master_clean.csv")
nifty50_df = pd.read_csv(PROCESSED_DIR / "nifty50.csv", parse_dates=["Date"])
nifty100_df = pd.read_csv(PROCESSED_DIR / "nifty100.csv", parse_dates=["Date"])

# Clean & normalize perf_df columns
perf_df.columns = perf_df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True)
perf_df.rename(columns={
    "returns_1m_": "returns_1m",
    "returns_3m_": "returns_3m",
    "returns_6m_": "returns_6m",
    "returns_1y_": "returns_1y",
    "returns_3y_": "returns_3y",
    "returns_5y_": "returns_5y",
    "expense_ratio_": "expense_ratio",
    "aum_cr_": "aum_cr"
}, inplace=True)

# Set index for easy matching
nifty50_df.set_index("Date", inplace=True)
nifty100_df.set_index("Date", inplace=True)

print(f"nav_df       : {nav_df.shape} rows")
print(f"perf_df      : {perf_df.shape} rows")
print(f"master_df    : {master_df.shape} rows")
print(f"nifty50_df   : {nifty50_df.shape} rows")
print(f"nifty100_df  : {nifty100_df.shape} rows")


Loading datasets...
nav_df       : (10980, 14) rows
perf_df      : (120, 21) rows
master_df    : (10, 12) rows
nifty50_df   : (1097, 5) rows
nifty100_df  : (1097, 5) rows


## 3. Daily Returns Computation

In [3]:
# ============================================================
# DAILY RETURNS COMPUTATION
# ============================================================
# Sort data chronologically
nav_df.sort_values(["scheme_code", "nav_date"], inplace=True)

# Calculate daily return for each scheme
nav_df["daily_return"] = nav_df.groupby("scheme_code")["nav"].pct_change()

# Calculate daily return for Nifty 100 (benchmark)
nifty100_df.sort_index(inplace=True)
nifty100_df["daily_return"] = nifty100_df["Close"].pct_change()

# Calculate daily return for Nifty 50
nifty50_df.sort_index(inplace=True)
nifty50_df["daily_return"] = nifty50_df["Close"].pct_change()

print("Daily returns calculated.")
print("Example scheme daily returns:")
print(nav_df[["scheme_name", "nav_date", "nav", "daily_return"]].dropna().head(3))


Daily returns calculated.
Example scheme daily returns:
                                            scheme_name  ... daily_return
5885                Kotak Bluechip Fund - Direct Growth  ...    -0.780685
4877   ICICI Prudential Technology Fund - Direct Growth  ...     3.345658
10449       Parag Parikh Flexi Cap Fund - Direct Growth  ...     0.280248

[3 rows x 4 columns]


## 4. CAGR (Compounded Annual Growth Rate) Calculations
We calculate CAGR for **1-Year** (representing 2024 calendar returns) and **3-Year** (representing 2022–2024 returns). 
Since the available daily history covers 3 years, the **5-Year CAGR** is retrieved from the processed `scheme_performance_clean.csv` dataset, which contains historical performance parameters.


In [4]:
# ============================================================
# CAGR CALCULATIONS (1Y, 3Y, 5Y)
# ============================================================

cagr_results = []
unique_schemes = nav_df["scheme_code"].unique()

for sc in unique_schemes:
    s_nav = nav_df[nav_df["scheme_code"] == sc].copy()
    s_name = s_nav["scheme_name"].iloc[0]
    
    # 3-Year CAGR (Full historical range 2022-01-03 to 2024-12-31)
    s_nav.sort_values("nav_date", inplace=True)
    first_nav_3y = s_nav["nav"].iloc[0]
    last_nav_3y = s_nav["nav"].iloc[-1]
    days_3y = (s_nav["nav_date"].iloc[-1] - s_nav["nav_date"].iloc[0]).days
    cagr_3y = (last_nav_3y / first_nav_3y) ** (365.25 / days_3y) - 1
    
    # 1-Year CAGR (2024 calendar year)
    s_nav_2024 = s_nav[s_nav["nav_date"].dt.year == 2024]
    if not s_nav_2024.empty:
        first_nav_1y = s_nav_2024["nav"].iloc[0]
        last_nav_1y = s_nav_2024["nav"].iloc[-1]
        days_1y = (s_nav_2024["nav_date"].iloc[-1] - s_nav_2024["nav_date"].iloc[0]).days
        cagr_1y = (last_nav_1y / first_nav_1y) ** (365.25 / days_1y) - 1
    else:
        cagr_1y = np.nan
        
    # 5-Year CAGR (from scheme_performance_clean.csv)
    p_row = perf_df[perf_df["scheme_code"] == sc]
    if not p_row.empty:
        cagr_5y = p_row["returns_5y"].iloc[0] / 100.0  # Convert from percentage
    else:
        cagr_5y = np.nan
        
    cagr_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "cagr_1y": cagr_1y,
        "cagr_3y": cagr_3y,
        "cagr_5y": cagr_5y
    })

cagr_df = pd.DataFrame(cagr_results)
print("CAGR calculations complete:")
cagr_df.head(10)


CAGR calculations complete:


## 5. Sharpe Ratio Computation
The Sharpe Ratio measures the excess return per unit of total risk:
$$\text{Sharpe Ratio} = \frac{R_p - R_f}{\sigma_p} \times \sqrt{252}$$
Where:
- $R_p$ is the daily return of the portfolio (scheme).
- $R_f$ is the daily risk-free rate ($6.5\% / 252$).
- $\sigma_p$ is the standard deviation of daily scheme returns.


In [5]:
# ============================================================
# SHARPE RATIO COMPUTATION
# ============================================================
daily_rf = RISK_FREE_RATE_ANN / TRADING_DAYS_YEAR
sharpe_results = []

for sc in unique_schemes:
    s_returns = nav_df[nav_df["scheme_code"] == sc]["daily_return"].dropna()
    s_name = nav_df[nav_df["scheme_code"] == sc]["scheme_name"].iloc[0]
    
    # Calculate excess returns
    excess_returns = s_returns - daily_rf
    mean_excess = excess_returns.mean()
    std_returns = s_returns.std()
    
    # Annualized Sharpe
    if std_returns > 0:
        sharpe = (mean_excess / std_returns) * np.sqrt(TRADING_DAYS_YEAR)
    else:
        sharpe = np.nan
        
    sharpe_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "sharpe_ratio": sharpe
    })

sharpe_df = pd.DataFrame(sharpe_results)

# Export Sharpe ratio report
sharpe_df.to_csv(REPORTS_DIR / "sharpe_ratio.csv", index=False)
print("Sharpe ratio calculations complete and saved to reports/sharpe_ratio.csv:")
sharpe_df.head(10)


Sharpe ratio calculations complete and saved to reports/sharpe_ratio.csv:


## 6. Sortino Ratio Computation
The Sortino Ratio is similar to the Sharpe Ratio, but it only penalizes downside volatility:
$$\text{Sortino Ratio} = \frac{R_p - R_f}{\sigma_d} \times \sqrt{252}$$
Where $\sigma_d$ is the **downside semi-deviation** (standard deviation of daily returns falling below the risk-free rate):
$$\sigma_d = \sqrt{\frac{1}{N} \sum_{t=1}^N \min(R_{p,t} - R_f, 0)^2}$$


In [6]:
# ============================================================
# SORTINO RATIO COMPUTATION
# ============================================================
sortino_results = []

for sc in unique_schemes:
    s_returns = nav_df[nav_df["scheme_code"] == sc]["daily_return"].dropna()
    s_name = nav_df[nav_df["scheme_code"] == sc]["scheme_name"].iloc[0]
    
    # Calculate daily excess returns
    excess_returns = s_returns - daily_rf
    mean_excess = excess_returns.mean()
    
    # Downside deviations (only negative excess returns, square differences)
    downside_diffs = np.minimum(excess_returns, 0.0)
    downside_var = np.mean(downside_diffs ** 2)
    downside_dev = np.sqrt(downside_var)
    
    # Annualized Sortino
    if downside_dev > 0:
        sortino = (mean_excess / downside_dev) * np.sqrt(TRADING_DAYS_YEAR)
    else:
        sortino = np.nan
        
    sortino_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "sortino_ratio": sortino
    })

sortino_df = pd.DataFrame(sortino_results)

# Export Sortino ratio report
sortino_df.to_csv(REPORTS_DIR / "sortino_ratio.csv", index=False)
print("Sortino ratio calculations complete and saved to reports/sortino_ratio.csv:")
sortino_df.head(10)


Sortino ratio calculations complete and saved to reports/sortino_ratio.csv:


## 7. Alpha & Beta Computation (against Nifty 100)
Under the Capital Asset Pricing Model (CAPM), we calculate:
- **Beta ($\beta$)**: Measures systemic risk (relative sensitivity to the benchmark Nifty 100).
- **Annualized Alpha ($\alpha$)**: Measures manager skill (annualized excess return over risk-adjusted benchmark expectation).
We regress daily excess returns of the scheme against daily excess returns of the Nifty 100 index:
$$R_p - R_f = \alpha + \beta (R_b - R_f)$$
Annualized Alpha is computed as:
$$\alpha_{\text{annual}} = \alpha_{\text{daily}} \times 252$$


In [7]:
# ============================================================
# ALPHA & BETA COMPUTATION (CAPM vs Nifty 100)
# ============================================================
# Group benchmark daily returns to match
n100_ret = nifty100_df["daily_return"].dropna()

alpha_beta_results = []

for sc in unique_schemes:
    s_data = nav_df[nav_df["scheme_code"] == sc][["nav_date", "daily_return"]].dropna()
    s_name = nav_df[nav_df["scheme_code"] == sc]["scheme_name"].iloc[0]
    
    # Merge on date to ensure alignment
    merged = pd.merge(s_data, n100_ret.to_frame("nifty100_return"), left_on="nav_date", right_index=True)
    
    if len(merged) > 30:
        # Calculate daily excess returns
        excess_scheme = merged["daily_return"] - daily_rf
        excess_benchmark = merged["nifty100_return"] - daily_rf
        
        # Linear Regression using scipy.stats.linregress
        slope, intercept, r_value, p_value, std_err = stats.linregress(excess_benchmark, excess_scheme)
        
        beta = slope
        daily_alpha = intercept
        
        # Annualized Alpha (simple multiplication is standard, but compounding is also useful)
        alpha_ann = daily_alpha * TRADING_DAYS_YEAR
        r_squared = r_value ** 2
    else:
        beta = np.nan
        alpha_ann = np.nan
        r_squared = np.nan
        
    alpha_beta_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "beta": beta,
        "alpha_annual": alpha_ann,
        "r_squared": r_squared
    })

ab_df = pd.DataFrame(alpha_beta_results)

# Export Alpha & Beta report
ab_df.to_csv(REPORTS_DIR / "alpha_beta.csv", index=False)
print("Alpha & Beta calculations complete and saved to reports/alpha_beta.csv:")
ab_df.head(10)


Alpha & Beta calculations complete and saved to reports/alpha_beta.csv:


## 8. Maximum Drawdown Calculation
Maximum Drawdown (MDD) represents the largest peak-to-trough drop in NAV value over the full period:
$$\text{Drawdown}_t = \frac{\text{Peak}_t - \text{NAV}_t}{\text{Peak}_t}, \quad \text{MDD} = \max(\text{Drawdown}_t)$$
Where $\text{Peak}_t = \max_{\tau \le t}(\text{NAV}_\tau)$.


In [8]:
# ============================================================
# MAXIMUM DRAWDOWN COMPUTATION
# ============================================================
mdd_results = []

for sc in unique_schemes:
    s_nav = nav_df[nav_df["scheme_code"] == sc].copy()
    s_nav.sort_values("nav_date", inplace=True)
    s_name = s_nav["scheme_name"].iloc[0]
    
    nav_series = s_nav["nav"]
    
    # Running cumulative max peak
    peaks = nav_series.cummax()
    
    # Drawdowns
    drawdowns = (nav_series - peaks) / peaks
    max_dd = drawdowns.min()
    
    mdd_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "max_drawdown": max_dd
    })

mdd_df = pd.DataFrame(mdd_results)
print("Maximum Drawdown calculations complete:")
mdd_df.head(10)


Maximum Drawdown calculations complete:


## 9. Tracking Error Computation
Tracking Error measures the standard deviation of active returns (difference between scheme daily return and Nifty 100 benchmark return) annualized:
$$\text{Tracking Error} = \sigma(R_p - R_b) \times \sqrt{252}$$


In [9]:
# ============================================================
# TRACKING ERROR COMPUTATION
# ============================================================
te_results = []

for sc in unique_schemes:
    s_data = nav_df[nav_df["scheme_code"] == sc][["nav_date", "daily_return"]].dropna()
    s_name = nav_df[nav_df["scheme_code"] == sc]["scheme_name"].iloc[0]
    
    # Merge with benchmark daily returns
    merged = pd.merge(s_data, n100_ret.to_frame("nifty100_return"), left_on="nav_date", right_index=True)
    
    if len(merged) > 30:
        # Calculate active returns (tracking difference)
        active_returns = merged["daily_return"] - merged["nifty100_return"]
        
        # Tracking Error is standard deviation of active returns annualized
        tracking_error = active_returns.std() * np.sqrt(TRADING_DAYS_YEAR)
    else:
        tracking_error = np.nan
        
    te_results.append({
        "scheme_code": sc,
        "scheme_name": s_name,
        "tracking_error": tracking_error
    })

te_df = pd.DataFrame(te_results)
print("Tracking Error calculations complete:")
te_df.head(10)


Tracking Error calculations complete:


## 10. Weighted Fund Scorecard Creation
We create a consolidated Fund Scorecard by ranking the 10 mutual funds across 5 key performance metrics:
- **30%**: 3Y CAGR Return Rank (Higher is better)
- **25%**: Sharpe Ratio Rank (Higher is better)
- **20%**: Alpha Rank (Higher is better)
- **15%**: Expense Ratio Rank (Lower is better / Inverse rank)
- **10%**: Max Drawdown Rank (Lower absolute drawdown is better / Inverse rank)

For each metric, funds are assigned a rank from 1 (best) to 10 (worst). The final score is:
$$\text{Score} = 0.30 \times R_{3Y} + 0.25 \times R_{\text{Sharpe}} + 0.20 \times R_{\text{Alpha}} + 0.15 \times R_{\text{Expense}} + 0.10 \times R_{\text{Drawdown}}$$
A lower score indicates a superior overall rating.


In [10]:
# ============================================================
# WEIGHTED FUND SCORECARD CREATION
# ============================================================
# Consolidate all metrics into one DataFrame
scorecard = cagr_df[["scheme_code", "scheme_name", "cagr_3y"]].copy()

# Add Sharpe, Alpha, Max Drawdown
scorecard = pd.merge(scorecard, sharpe_df[["scheme_code", "sharpe_ratio"]], on="scheme_code")
scorecard = pd.merge(scorecard, ab_df[["scheme_code", "alpha_annual", "beta"]], on="scheme_code")
scorecard = pd.merge(scorecard, mdd_df[["scheme_code", "max_drawdown"]], on="scheme_code")
scorecard = pd.merge(scorecard, te_df[["scheme_code", "tracking_error"]], on="scheme_code")

# Merge expense ratio from database/performance dataset
# latest_perf_df view/table is available from Day 3
# Since it is also in perf_df, let's extract the latest expense ratio per scheme
latest_perf = perf_df.sort_values("as_of_date").groupby("scheme_code").last().reset_index()
scorecard = pd.merge(scorecard, latest_perf[["scheme_code", "expense_ratio"]], on="scheme_code")

# Ranks (1 is best, 10 is worst)
scorecard["rank_returns"] = scorecard["cagr_3y"].rank(ascending=False, method="min")
scorecard["rank_sharpe"]  = scorecard["sharpe_ratio"].rank(ascending=False, method="min")
scorecard["rank_alpha"]   = scorecard["alpha_annual"].rank(ascending=False, method="min")

# Lower expense ratio is better
scorecard["rank_expense"] = scorecard["expense_ratio"].rank(ascending=True, method="min")

# Max drawdown is negative. Less negative (closer to 0) is better. 
# So sorting descending ranks the closest to 0 (highest value) as 1.
scorecard["rank_drawdown"] = scorecard["max_drawdown"].rank(ascending=False, method="min")

# Calculate weighted score
scorecard["scorecard_score"] = (
    0.30 * scorecard["rank_returns"] +
    0.25 * scorecard["rank_sharpe"] +
    0.20 * scorecard["rank_alpha"] +
    0.15 * scorecard["rank_expense"] +
    0.10 * scorecard["rank_drawdown"]
)

# Final rank based on score
scorecard["final_scorecard_rank"] = scorecard["scorecard_score"].rank(ascending=True, method="min")

# Sort by final rank
scorecard.sort_values("final_scorecard_rank", inplace=True)

# Save to CSV
scorecard.to_csv(REPORTS_DIR / "fund_scorecard.csv", index=False)
print("Weighted Fund Scorecard complete and saved to reports/fund_scorecard.csv:")
scorecard[["scheme_name", "cagr_3y", "sharpe_ratio", "alpha_annual", "expense_ratio", "max_drawdown", "scorecard_score", "final_scorecard_rank"]].head(10)


Weighted Fund Scorecard complete and saved to reports/fund_scorecard.csv:


## 11. Benchmark Comparison Visualizations
We plot the cumulative growth of a hypothetical investment of **INR 100** in each of the 10 mutual fund schemes vs the **Nifty 100 index** over the 3-year history (2022–2024).


In [11]:
# ============================================================
# BENCHMARK COMPARISON PLOT
# ============================================================
# Calculate cumulative growth series starting at 100
cum_df = pd.pivot_table(nav_df, index="nav_date", columns="scheme_name", values="nav")
cum_df = cum_df.ffill().bfill()

# Normalize each column to start at 100
cum_growth = (cum_df / cum_df.iloc[0]) * 100.0

# Add Nifty 100 benchmark
n100_close = nifty100_df["Close"].ffill().bfill()
n100_normalized = (n100_close / n100_close.iloc[0]) * 100.0
cum_growth["NIFTY 100 Benchmark"] = n100_normalized

# 1. Plotly Interactive Chart
fig = go.Figure()
for col in cum_growth.columns:
    is_bench = col == "NIFTY 100 Benchmark"
    fig.add_trace(go.Scatter(
        x=cum_growth.index,
        y=cum_growth[col],
        name=col.split("-")[0].strip()[:25],
        line=dict(
            width=3.0 if is_bench else 1.2,
            color="#FF4B4B" if is_bench else None,
            dash="dash" if is_bench else "solid"
        ),
        hovertemplate="<b>%{x|%Y-%m-%d}</b><br>" + col[:20] + ": INR %{y:.2f}<extra></extra>"
    ))

fig.update_layout(
    title="Cumulative Growth of INR 100 Investment (2022–2024) vs Nifty 100",
    xaxis_title="Date",
    yaxis_title="Portfolio Value (INR)",
    template=PLOTLY_TEMPLATE,
    height=600,
    legend=dict(orientation="v", x=1.01, y=1),
    hovermode="x unified"
)
fig.show()

# 2. Matplotlib/Seaborn High-Resolution Static Image Export
plt.figure(figsize=(14, 7), facecolor="#1e1e2e")
ax = plt.gca()
ax.set_facecolor("#2a2a3e")
ax.grid(True, color="#3a3a55", linestyle=":", alpha=0.6)

# Plot schemes
for col in cum_growth.columns:
    if col != "NIFTY 100 Benchmark":
        ax.plot(cum_growth.index, cum_growth[col], label=col.split("-")[0].strip()[:20], alpha=0.7, linewidth=1.2)

# Plot benchmark distinctly
ax.plot(cum_growth.index, cum_growth["NIFTY 100 Benchmark"], label="NIFTY 100 Benchmark", color="red", linestyle="--", linewidth=2.5)

ax.set_title("Benchmark Comparison — Scheme Performance vs Nifty 100 (2022–2024)", color="white", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Date", color="lightgray", fontsize=11)
ax.set_ylabel("Hypothetical Investment Value (Start = 100)", color="lightgray", fontsize=11)
ax.tick_params(colors="lightgray", labelsize=10)
for spine in ax.spines.values():
    spine.set_edgecolor("#444466")

ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', facecolor="#2a2a3e", edgecolor="#444466", labelcolor="white", fontsize=9)

# Save high-res PNG
fig_path = CHARTS_DIR / "benchmark_comparison.png"
# Also save directly inside reports/ for convenience
fig_path_direct = REPORTS_DIR / "benchmark_comparison.png"

plt.savefig(str(fig_path), dpi=150, bbox_inches="tight", facecolor="#1e1e2e")
plt.savefig(str(fig_path_direct), dpi=150, bbox_inches="tight", facecolor="#1e1e2e")
plt.close()

print(f"Static chart saved successfully to: {fig_path.name}")


Static chart saved successfully to: benchmark_comparison.png


## 12. Day 4 Analytical Summary & Performance Insights

| Fund Name | 3Y CAGR | Sharpe Ratio | CAPM Beta | Annualized Alpha | Max Drawdown | Tracking Error | Scorecard Rank |
|---|---|---|---|---|---|---|---|
| *Ranked outputs loaded dynamically in CSV files* | | | | | | | |

### Key Analyst Observations:
1. **Beta Analysis**: Schemes with Betas close to 1.0 (e.g. Large Cap funds) exhibit systemic market risk tracking. Mid-cap and Small-cap funds show Betas greater than 1.0, denoting higher sensitivity.
2. **Alpha (CAPM)**: Positive Annualized Alphas indicate active management skill yielding returns above risk-adjusted expectations. Mid-cap and Small-cap funds have generated substantial Alpha during the 2023–2024 bull cycles.
3. **Sharpe & Sortino comparison**: Ratios above 1.0 indicate strong risk-adjusted returns. High Sortino relative to Sharpe suggests the fund has successfully mitigated large downside spikes, which is a desirable management signature.
4. **Drawdown Resilience**: Maximum Drawdown measures historical risk. Conservative and large-cap funds show shallower drawdowns compared to mid/small-cap categories, demonstrating capital preservation strength during market corrections.
5. **Weighted Leaderboard**: The scorecard successfully incorporates both returns (CAGR), manager skill (Alpha), volatility adjusted risk (Sharpe), cost structure (Expense ratio), and loss history (Drawdown) to rank the funds objectively.
